# Task1

1. What is a Text-to-Math problem?

A Text-to-Math problem is a word-based mathematical question where numerical relationships are described in natural language and must be converted into mathematical operations to compute an answer.

2. Why are agents useful for math reasoning?

Agents:

Break problems into explicit steps

Decide when to calculate

Use tools (like calculators) for accurate computation

Reduce hallucination compared to raw LLM responses

3. Difference:

Normal LLM :  Responds in one pass 
may guess calculations 
no memory 
less reliable for math 
Agent based Reasoning: thinks step-by-step 
uses tool for math 
can track context
more accrurate


# Task2

In [10]:
%pip install -U langchain-experimental -q


Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install -U langchain langchain-community langchain-experimental transformers -q


Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.4.1 which is incompatible.
sentence-transformers 3.0.1 requires transformers<5.0.0,>=4.34.0, but you have transformers 5.1.0 which is incompatible.


In [7]:
from langchain_experimental.tools import PythonREPLTool
from langchain_core.tools import Tool
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from dotenv import load_dotenv
from langsmith import Client
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent
load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)


# Tool
python_tool = PythonREPLTool()
tools = [
    Tool(
        name="Calculator",
        func=python_tool.run,
        description="Use this tool for math calculations",
    )
]


# Explicit ReAct prompt (required)
prompt = PromptTemplate.from_template(
    """Answer the following question as best you can.
You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: think step-by-step
Action: one of [{tool_names}]
Action Input: input to the action
Observation: result
Thought: I now know the final answer
Final Answer: the final answer

{agent_scratchpad}
"""
)


# Agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

math_agent = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)


In [8]:
result = math_agent.invoke(
    {"input": "What is 15% of 320?"}
)

print(result)




> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


Question: What is 15% of 320?
Thought: think step-by-step
To find 15% of 320, I need to multiply 320 by 0.15.

Action: Calculator
Action Input: input to the action = 320 * 0.15SyntaxError('invalid syntax', ('<string>', 1, 7, 'input to the action = 320 * 0.15\n', 1, 9))Question: What is 15% of 320?
Thought: think step-by-step
To find 15% of 320, I need to multiply 320 by 0.15.

Action: Calculator
Action Input: input to the action = 320 * 0.15SyntaxError('invalid syntax', ('<string>', 1, 7, 'input to the action = 320 * 0.15\n', 1, 9))Question: What is 15% of 320?
Thought: think step-by-step
To find 15% of 320, I need to multiply 320 by 0.15.

Action: Calculator
Action Input: input to the action = 320 * 0.15SyntaxError('invalid syntax', ('<string>', 1, 7, 'input to the action = 320 * 0.15\n', 1, 9))Question: What is 15% of 320?
Thought: think step-by-step
To find 15% of 320, I need to multiply 320 by 0.15.

Action: Calculator
Action Input: input to the action = 320 * 0.15SyntaxError('inva

# Task3

In [ ]:
import streamlit as st

st.title("🧮 Text-to-Math Agent")

if "history" not in st.session_state:
    st.session_state.history = []

user_input = st.text_input("Enter a math problem:")

if st.button("Solve") and user_input:
    answer = math_agent.run(user_input)

    st.session_state.history.append(
        {"question": user_input, "answer": answer}
    )

for item in st.session_state.history:
    st.markdown(f"**Q:** {item['question']}")
    st.markdown(f"**A:** {item['answer']}")
    st.divider()